# **Catch Me If You Can Network Analysis**

**Group:** H

**Group members:**  
- Andrea Cipolla — ID 319211  
- Arianna Cambi — ID 317971  
- Sofia Capriolo — ID 309371  
- Giorgio Vanini — ID 311581  
- Maiia Kopalina — ID 321891  

## **Introduction**

This project applies **Social Network Analysis (SNA)** to the movie *Catch Me If You Can*. The objective is to study the structure of interactions between characters by representing them as a network and analyzing its structural properties using computational tools.

**Network:** *Catch Me If You Can* movie network  
**Type:** undirected, weighted social network  
**Meaning of nodes:** characters appearing in the movie  
**Meaning of edges:** two characters appear in the same scene  
**Meaning of weights:** number of scenes in which the two characters appear together

**Source:** J. Kaminski et al., *Moviegalaxies - Social Networks in Movies*, Harvard Dataverse, V3 (2018).

### **Project Goals**

In this representation, characters are modeled as nodes in a network, while their interactions are captured through edges connecting characters who appear in the same scene. The weight associated with each edge reflects the intensity of interaction, measured by the number of shared-scene appearances.

The aim of the project is to explore the structure of this network through several Social Network Analysis techniques, including:
- graph construction and visualization
- computation of basic network statistics
- clustering analysis and clustering distributions
- centrality measures
- community detection methods

### **Structure of the Analysis**

The analysis is organized following the weekly assignments of the course:
- **Week 1:** graph implementation, visualization, and computation of basic statistics
- **Week 2:** computation of clustering coefficients and comparison with built-in functions
- **Week 3:** analysis of cumulative distributions related to clustering
- **Week 5:** identification of the most central nodes using centrality measures
- **Week 6:** detection of communities using different algorithms and comparison of their results


### **Background**

Although the dataset represents a fictional narrative, it provides a meaningful example of a social structure where certain characters play central roles in connecting different parts of the story while others remain more peripheral.

The analysis is organized according to the weekly tasks assigned during the course. First, the graph is implemented and described through basic network statistics such as number of nodes, number of edges, average degree, and density. Then, the analysis focuses on local cohesion by studying clustering coefficients and their distributions. Next, centrality measures are used to identify the most important characters in the network. Finally, community detection methods are applied to identify groups of characters that form tightly connected substructures within the network.

Overall, this project combines computational implementation and interpretation, showing how network-based methods can reveal meaningful insights about the internal structure of a narrative system.


## Week 1

In this first part, we load the selected social network and represent it as a graph in Python using the NetworkX library. We then visualize the network and compute basic descriptive measures such as the number of nodes, edges, average degree, and density.

In [ ]:
from pathlib import Path
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# allow display both in notebooks and scripts
try:
    from IPython.display import display
    from IPython import get_ipython
except ImportError:
    def display(obj):
        print(obj)

    def get_ipython():
        return None

ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("matplotlib", "inline")  # show plots inside notebook

# plotting style
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)

# dataset paths
NODES_PATH = Path("datasets/nodes.csv")
EDGES_PATH = Path("datasets/edges.csv")

# load node and edge tables
nodes_df = pd.read_csv(NODES_PATH)
edges_df = pd.read_csv(EDGES_PATH)

# clean column names (remove extra spaces)
nodes_df.columns = [col.strip() for col in nodes_df.columns]
edges_df.columns = [col.strip() for col in edges_df.columns]

# create empty graph
G = nx.Graph()

# add nodes with character labels
for _, row in nodes_df.iterrows():
    G.add_node(int(row["# index"]), label=str(row["label"]).strip())

# add edges with weights (scene co-occurrence strength)
for _, row in edges_df.iterrows():
    G.add_edge(int(row["# source"]), int(row["target"]), weight=float(row["weight"]))

# quick check of graph size
print(f"Nodes loaded: {G.number_of_nodes()}")
print(f"Edges loaded: {G.number_of_edges()}")

# preview of datasets
display(nodes_df.head())
display(edges_df.head())

### Graph visualization

The dataset available in this folder does not include predefined plotting coordinates, so the graph is drawn with a force-directed layout (`spring_layout`). This still provides a valid visual overview of the network structure.

In [ ]:
plt.figure(figsize=(14, 10))

# spring layout to position nodes
layout = nx.spring_layout(G, seed=42, k=0.45)

# weighted degree used to scale node sizes
weighted_degree = dict(G.degree(weight='weight'))
node_sizes = [80 + 35 * weighted_degree[node] for node in G.nodes()]

# draw edges
nx.draw_networkx_edges(G, pos=layout, alpha=0.35, edge_color='gray')

# draw nodes
nx.draw_networkx_nodes(G, pos=layout, node_size=node_sizes, node_color='steelblue', alpha=0.9)

# add character labels
nx.draw_networkx_labels(
    G,
    pos=layout,
    labels={node: data['label'] for node, data in G.nodes(data=True)},
    font_size=8,
)

plt.title('Catch Me If You Can character network')
plt.axis('off')
plt.show()

### Basic network statistics

In [ ]:
# basic network statistics
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# average degree of the network
average_degree = sum(dict(G.degree()).values()) / num_nodes

# network density
density = nx.density(G)

summary_week1 = pd.DataFrame(
    {
        'metric': ['Number of nodes', 'Number of edges', 'Average degree', 'Density'],
        'value': [num_nodes, num_edges, average_degree, density],
    }
)

summary_week1

### Comment
The computed statistics provide a first quantitative overview of the network structure. The graph contains **82 nodes and 162 edges**, indicating a moderate number of characters but a relatively limited number of interactions between them. The **average degree of approximately 3.95** means that each character is connected, on average, to about **four others** through shared scenes. The **density of about 0.049** confirms that the network is quite sparse, meaning that only a small fraction of all possible connections actually occur. This pattern is typical of character interaction networks, where most characters interact with a limited subset of the cast rather than with every other character.

## Week 2

In this part, we restrict the analysis to the **largest connected component** of the network and study its local cohesion through clustering measures. We implement a custom function to compute the clustering coefficient of each node, then use it to calculate the average clustering and compare the result with the corresponding built-in NetworkX measures.

In [ ]:
# work on the largest connected component
if nx.is_connected(G):
    G_lcc = G.copy()
else:
    largest_component_nodes = max(nx.connected_components(G), key=len)
    G_lcc = G.subgraph(largest_component_nodes).copy()

print(f'Largest connected component: {G_lcc.number_of_nodes()} nodes, {G_lcc.number_of_edges()} edges')

In [ ]:
# manual computation of node clustering coefficients
def local_clustering_manual(graph):
    clustering = {}

    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        k = len(neighbors)

        # nodes with fewer than two neighbors cannot form triangles
        if k < 2:
            clustering[node] = 0.0
            continue

        closed_triplets = 0
        for u, v in combinations(neighbors, 2):
            if graph.has_edge(u, v):
                closed_triplets += 1

        # clustering coefficient formula
        clustering[node] = closed_triplets / (k * (k - 1) / 2)

    return clustering


# compute average clustering manually
def average_clustering_manual(graph):
    clustering_values = local_clustering_manual(graph)
    return sum(clustering_values.values()) / len(clustering_values)


# manual measures
manual_clustering = local_clustering_manual(G_lcc)
manual_average_clustering = average_clustering_manual(G_lcc)

# NetworkX built-in measures for comparison
networkx_average_clustering = nx.average_clustering(G_lcc)
networkx_transitivity = nx.transitivity(G_lcc)

comparison_week2 = pd.DataFrame(
    {
        'measure': [
            'Manual average clustering',
            'NetworkX average clustering',
            'NetworkX transitivity',
        ],
        'value': [
            manual_average_clustering,
            networkx_average_clustering,
            networkx_transitivity,
        ],
    }
)

comparison_week2

In [ ]:
# create a table of nodes with their clustering coefficients
clustering_df = pd.DataFrame(
    {
        'node': list(manual_clustering.keys()),
        'label': [G_lcc.nodes[node]['label'] for node in manual_clustering.keys()],
        'clustering': list(manual_clustering.values()),
    }
).sort_values('clustering', ascending=False)

# show the nodes with the highest clustering
clustering_df.head(10)

### Comment

The results confirm that the custom implementation is correct, since the **manual average clustering** exactly matches the value returned by `nx.average_clustering()`, both equal to about **0.599**. This indicates a relatively high tendency of characters to form locally connected groups, meaning that if two characters are connected to the same one, they are often also connected to each other. By contrast, the **transitivity** is much lower, about **0.115**, because it is a global measure based on all triplets in the network rather than the average of local node-level coefficients. The difference between the two values suggests that triangle closure is not distributed uniformly across all characters, but is stronger around some nodes than others.

## Week 3

In this part, we analyze how clustering is distributed across the largest connected component. We first compute the **cumulative distribution of node clustering coefficients**, then define for each node the average clustering of its neighbors and compute the cumulative distribution of this second quantity in order to compare the two patterns.

In [ ]:
# empirical cumulative distribution function
def empirical_cdf(values):
    values = np.sort(np.asarray(values, dtype=float))
    cumulative = np.arange(1, len(values) + 1) / len(values)
    return values, cumulative


# clustering values of nodes
clustering_values = list(manual_clustering.values())

# compute CDF of clustering coefficients
clustering_x, clustering_cdf = empirical_cdf(clustering_values)

In [ ]:
# compute average clustering of neighbors for each node
def average_neighbor_clustering(graph, clustering_by_node):
    neighbor_average = {}

    for node in graph.nodes():
        neighbors = list(graph.neighbors(node))
        if not neighbors:
            neighbor_average[node] = 0.0
            continue

        # average clustering among neighbors
        neighbor_average[node] = float(np.mean([clustering_by_node[neighbor] for neighbor in neighbors]))

    return neighbor_average


# compute neighbor clustering values
neighbor_clustering = average_neighbor_clustering(G_lcc, manual_clustering)

neighbor_values = list(neighbor_clustering.values())

# CDF of neighbors' average clustering
neighbor_x, neighbor_cdf = empirical_cdf(neighbor_values)

# table of nodes ranked by neighbors' clustering
neighbor_clustering_df = pd.DataFrame(
    {
        'node': list(neighbor_clustering.keys()),
        'label': [G_lcc.nodes[node]['label'] for node in neighbor_clustering.keys()],
        'avg_neighbor_clustering': list(neighbor_clustering.values()),
    }
).sort_values('avg_neighbor_clustering', ascending=False)

neighbor_clustering_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].step(clustering_x, clustering_cdf, where='post', color='darkorange')
axes[0].set_title('CDF of node clustering')
axes[0].set_xlabel('Clustering coefficient')
axes[0].set_ylabel('Cumulative probability')

axes[1].step(neighbor_x, neighbor_cdf, where='post', color='teal')
axes[1].set_title('CDF of average neighbor clustering')
axes[1].set_xlabel('Average clustering of neighbors')
axes[1].set_ylabel('Cumulative probability')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.step(clustering_x, clustering_cdf, where='post', label='Node clustering', linewidth=2)
plt.step(neighbor_x, neighbor_cdf, where='post', label='Average neighbor clustering', linewidth=2)
plt.xlabel('Clustering value')
plt.ylabel('Cumulative probability')
plt.title('Comparison of the two cumulative distributions')
plt.legend()
plt.show()

In [ ]:
# summary statistics for the comparison
clustering_values_df = clustering_df.copy()
avg_neighbor_clust_df = neighbor_clustering_df.copy()

comparison_summary = pd.DataFrame({
    "Measure": [
        "Mean node clustering",
        "Median node clustering",
        "Mean neighbors' average clustering",
        "Median neighbors' average clustering"
    ],
    "Value": [
        clustering_values_df["clustering"].mean(),
        clustering_values_df["clustering"].median(),
        avg_neighbor_clust_df["avg_neighbor_clustering"].mean(),
        avg_neighbor_clust_df["avg_neighbor_clustering"].median()
    ]
})

comparison_summary

### Comment

The comparison between the two cumulative distributions shows that **node clustering coefficients** are generally higher than the **average clustering of neighbors**. This is also confirmed by the summary statistics: the mean node clustering is about **0.599**, while the mean neighbors’ average clustering is only about **0.339**. In other words, many characters belong to locally cohesive groups, but their neighbors are not, on average, embedded in equally clustered neighborhoods. The higher median node clustering **(0.75)** reinforces the idea that triangle closure is relatively common around individual characters, whereas the lower median neighbor value **(about 0.348)** suggests a less cohesive local environment when looking one step away. Overall, the two distributions indicate that clustering is present in the network but is not distributed uniformly across neighborhoods.

## Week 5

In this part we analyze node importance using two centrality measures: **betweenness centrality** and **PageRank**. Betweenness identifies characters that act as bridges connecting different parts of the network, while PageRank measures global importance based on connections to already influential nodes. Since the graph is weighted, weights are handled carefully when computing these measures. We implement PageRank manually and compare the results with the built-in **NetworkX PageRank** function as a validation step before using the measure to analyze the network structure.


In [ ]:
# convert edge weights into distances for shortest-path calculations
for u, v, data in G.edges(data=True):
    data['distance'] = 1.0 / data['weight']


# manual implementation of weighted PageRank
def pagerank_manual(graph, alpha=0.85, max_iter=200, tol=1.0e-9):
    nodes = list(graph.nodes())
    n = len(nodes)

    # initial uniform rank
    rank = {node: 1.0 / n for node in nodes}

    # weighted degree (used to normalize transitions)
    weighted_degree = {
        node: sum(data.get('weight', 1.0) for _, _, data in graph.edges(node, data=True))
        for node in nodes
    }

    for _ in range(max_iter):
        new_rank = {node: (1.0 - alpha) / n for node in nodes}

        # handle dangling nodes
        dangling_sum = sum(rank[node] for node in nodes if weighted_degree[node] == 0)
        dangling_contribution = alpha * dangling_sum / n
        for node in nodes:
            new_rank[node] += dangling_contribution

        # distribute rank through outgoing edges
        for source in nodes:
            if weighted_degree[source] == 0:
                continue
            for target in graph.neighbors(source):
                weight = graph[source][target].get('weight', 1.0)
                new_rank[target] += alpha * rank[source] * (weight / weighted_degree[source])

        error = sum(abs(new_rank[node] - rank[node]) for node in nodes)
        rank = new_rank
        if error < tol:
            break

    return rank


# compute centrality measures
betweenness_scores = nx.betweenness_centrality(G, weight='distance')
pagerank_scores = pagerank_manual(G)

In [ ]:
# validate the manual PageRank against NetworkX
networkx_pagerank = nx.pagerank(G, weight='weight')

pagerank_validation = pd.DataFrame(
    {
        'node': list(G.nodes()),
        'label': [G.nodes[node]['label'] for node in G.nodes()],
        'manual_pagerank': [pagerank_scores[node] for node in G.nodes()],
        'networkx_pagerank': [networkx_pagerank[node] for node in G.nodes()],
    }
)

pagerank_validation['abs_difference'] = (
    pagerank_validation['manual_pagerank'] - pagerank_validation['networkx_pagerank']
).abs()

pagerank_validation.sort_values('abs_difference', ascending=False).head(10)

In [ ]:
# build table with centrality measures
centrality_df = pd.DataFrame(
    {
        'node': list(G.nodes()),
        'label': [G.nodes[node]['label'] for node in G.nodes()],
        'betweenness': [betweenness_scores[node] for node in G.nodes()],
        'pagerank': [pagerank_scores[node] for node in G.nodes()],
    }
)

# compute ranking positions
centrality_df['betweenness_rank'] = centrality_df['betweenness'].rank(ascending=False, method='min')
centrality_df['pagerank_rank'] = centrality_df['pagerank'].rank(ascending=False, method='min')

# combined ranking score
centrality_df['combined_rank_score'] = centrality_df['betweenness_rank'] + centrality_df['pagerank_rank']

centrality_df.sort_values('combined_rank_score').head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bet_top = centrality_df.sort_values('betweenness', ascending=False).head(10)
axes[0].barh(bet_top['label'][::-1], bet_top['betweenness'][::-1], color='indianred')
axes[0].set_title('Top 10 nodes by betweenness')
axes[0].set_xlabel('Betweenness centrality')

pr_top = centrality_df.sort_values('pagerank', ascending=False).head(10)
axes[1].barh(pr_top['label'][::-1], pr_top['pagerank'][::-1], color='slateblue')
axes[1].set_title('Top 10 nodes by PageRank')
axes[1].set_xlabel('PageRank score')

plt.tight_layout()
plt.show()

In [ ]:
# identify the most central nodes according to each measure
most_central_betweenness = centrality_df.sort_values('betweenness', ascending=False).iloc[0]
most_central_pagerank = centrality_df.sort_values('pagerank', ascending=False).iloc[0]

print(
    f"Highest betweenness: {most_central_betweenness['label']} "
    f"(node {int(most_central_betweenness['node'])}, score = {most_central_betweenness['betweenness']:.4f})"
)
print(
    f"Highest PageRank: {most_central_pagerank['label']} "
    f"(node {int(most_central_pagerank['node'])}, score = {most_central_pagerank['pagerank']:.4f})"
)

### Comment

The results show that **Frank Abagnale Jr.** is the most central character according to both measures. He has the **highest betweenness centrality**, meaning that many shortest paths between other characters pass through him, which reflects his role as the main connector across different parts of the story. He also obtains the highest **PageRank**, indicating that he is linked to many important characters and occupies a globally influential position in the network. The comparison with the built-in **NetworkX PageRank** confirms that the manual implementation produces almost identical values. Other characters such as **Joe Shaye** and **Frank Sr.** also appear among the top nodes, reinforcing the interpretation that Frank is the structural center of the movie’s character network.

### Advanced Analysis: PageRank vs Betweenness (Finding the "Brokers")

To further understand the role of characters in the network, we can compare their PageRank and Betweenness Centrality using a scatter plot. Characters that deviate significantly from the diagonal—specifically those with high Betweenness but low PageRank—act as crucial "brokers" or "bridges". In the context of *Catch Me If You Can*, these characters might appear in fewer scenes (low PageRank) but are essential for connecting different groups or plotlines (high Betweenness).

In [ ]:
plt.figure(figsize=(10, 6))

# Scatter plot of PageRank vs Betweenness
plt.scatter(centrality_df['pagerank'], centrality_df['betweenness'], color='steelblue', alpha=0.7, edgecolor='black', s=80)

# Add labels for the most interesting nodes (high betweenness or high pagerank)
for _, row in centrality_df.iterrows():
    if row['betweenness'] > 0.15 or row['pagerank'] > 0.05:
        plt.text(row['pagerank'] + 0.002, row['betweenness'] + 0.005, row['label'], fontsize=9)

plt.title('Character Roles: PageRank vs Betweenness Centrality')
plt.xlabel('PageRank (Global Influence)')
plt.ylabel('Betweenness Centrality (Bridge Role)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Interpretation:**
Characters in the upper-left area of the plot (if any) are the "Brokers"—they have high Betweenness but lower PageRank, meaning they connect different communities despite not being globally central. Conversely, those in the bottom-right are highly connected locally but do not act as bridges. As expected, **Frank** dominates both metrics, but this visualization helps spot secondary characters who hold the network together.

## Week 6

In this part, we study community structure after transforming the graph into an **undirected and unweighted** network, removing self-loops, and restricting attention to the **largest connected component**. We compare **bridge removal (Girvan–Newman)** and **modularity optimization**, then select the partition with the higher modularity and visualize it in ***Gephi.***

In [ ]:
# build an undirected, unweighted version of the graph
G_week6 = nx.Graph()
G_week6.add_nodes_from(G.nodes(data=True))
G_week6.add_edges_from((u, v) for u, v in G.edges() if u != v)

# work on the largest connected component
if nx.is_connected(G_week6):
    G_week6_lcc = G_week6.copy()
else:
    week6_component_nodes = max(nx.connected_components(G_week6), key=len)
    G_week6_lcc = G_week6.subgraph(week6_component_nodes).copy()

print(
    f"Week 6 graph: {G_week6_lcc.number_of_nodes()} nodes, "
    f"{G_week6_lcc.number_of_edges()} edges"
)

In [ ]:
import time

# run Girvan–Newman and keep the partition with highest modularity
def best_girvan_newman_partition(graph):

    start = time.perf_counter()

    best_partition = None
    best_modularity = float("-inf")
    modularity_trace = []

    for partition in nx.community.girvan_newman(graph):
        partition = tuple(frozenset(comm) for comm in partition)

        mod = nx.community.modularity(graph, partition)
        modularity_trace.append((len(partition), mod))

        if mod > best_modularity:
            best_modularity = mod
            best_partition = partition

        if len(partition) >= graph.number_of_nodes():
            break

    elapsed = time.perf_counter() - start
    return best_partition, best_modularity, modularity_trace, elapsed


# Girvan–Newman (bridge removal)
gn_partition, gn_modularity, gn_trace, gn_time = best_girvan_newman_partition(G_week6_lcc)


# greedy modularity optimization
start = time.perf_counter()
greedy_partition = list(nx.community.greedy_modularity_communities(G_week6_lcc))
greedy_modularity = nx.community.modularity(G_week6_lcc, greedy_partition)
greedy_time = time.perf_counter() - start


# comparison between the two methods
community_comparison = pd.DataFrame({
    "method": [
        "Bridge removal (Girvan–Newman)",
        "Modularity optimization (Greedy)"
    ],
    "num_communities": [len(gn_partition), len(greedy_partition)],
    "modularity": [gn_modularity, greedy_modularity],
    "time_seconds": [gn_time, greedy_time],
    "community_sizes": [
        sorted([len(c) for c in gn_partition], reverse=True),
        sorted([len(c) for c in greedy_partition], reverse=True),
    ],
})

community_comparison

In [ ]:
gn_trace_df = pd.DataFrame(gn_trace, columns=['num_communities', 'modularity'])

plt.figure(figsize=(8, 5))
plt.plot(gn_trace_df['num_communities'], gn_trace_df['modularity'], marker='o')
plt.axhline(greedy_modularity, color='firebrick', linestyle='--', label='Greedy modularity')
plt.xlabel('Number of communities in Girvan-Newman partition')
plt.ylabel('Modularity')
plt.title('Community detection comparison')
plt.legend()
plt.show()

In [ ]:
# select the partition with the highest modularity
if greedy_modularity >= gn_modularity:
    best_partition_name = 'Greedy modularity optimization'
    best_partition = greedy_partition
else:
    best_partition_name = 'Girvan-Newman'
    best_partition = gn_partition

# map each node to its community
partition_map = {}
for community_id, community_nodes in enumerate(best_partition):
    for node in community_nodes:
        partition_map[node] = community_id

# preview visualization of detected communities
preview_layout = nx.spring_layout(G_week6_lcc, seed=42)
node_colors = [partition_map[node] for node in G_week6_lcc.nodes()]

plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G_week6_lcc, pos=preview_layout, alpha=0.25, edge_color='gray')
nx.draw_networkx_nodes(
    G_week6_lcc,
    pos=preview_layout,
    node_color=node_colors,
    cmap=plt.cm.Set3,
    node_size=220,
)
nx.draw_networkx_labels(
    G_week6_lcc,
    pos=preview_layout,
    labels={node: G_week6_lcc.nodes[node]['label'] for node in G_week6_lcc.nodes()},
    font_size=7,
)
plt.title(f'Best partition preview: {best_partition_name}')
plt.axis('off')
plt.show()

In [ ]:
print(f'Best method: {best_partition_name}')
print('Community preview plotted successfully.')

In [ ]:
# Save the selected partition as a node attribute for Gephi
nx.set_node_attributes(G_week6_lcc, partition_map, "community")

# Export the graph for Gephi
nx.write_gexf(G_week6_lcc, "catch_me_if_you_can_week6.gexf")

print("GEXF file exported successfully.")

### Comment

The comparison shows that **modularity optimization** performs slightly better than **bridge removal**, with a modularity of about **0.432** compared with **0.414.** This suggests a partition with stronger within-community cohesion and weaker between-community connectivity. In addition, modularity optimization is much faster, which makes it more practical for this network. For these reasons, it is selected as the best method. The **Gephi visualization** helps make the detected communities more readable by separating groups spatially and coloring nodes according to their assigned community. The figure highlights a clear hub structure centered on **Frank**, while also showing distinct groups of secondary characters organized around different parts of the story.
